# Full Training Notebook
This notebook runs the full fine-tuning pipeline for Qwen2-VL using LoRA and the same collator used in `train.ipynb`. Adjust `training_args` below for longer runs.

### Dataset loading
This cell loads the dataset JSON files (`train.json`, `valid.json`, `test.json`) from the `weapon VLM dataset` directory into a `datasets.DatasetDict` named `dataset`. Run this first to make data available to the rest of the notebook.

In [8]:
from pathlib import Path
from datasets import load_dataset
DATA_DIR = Path("weapon VLM dataset")
dataset = load_dataset(
    "json",
    data_files={
        "train": str(DATA_DIR / "train.json"),
        "validation": str(DATA_DIR / "valid.json"),
        "test": str(DATA_DIR / "test.json"),
    },
)

### Model & LoRA setup
This cell loads the Qwen2-VL base model and `AutoProcessor`, configures 4-bit quantization, and wraps the model with LoRA (`get_peft_model`) using `lora_config`. Run after dataset is loaded to prepare `model` and `processor` for training.

In [9]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "Qwen/Qwen2-VL-2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv", "proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

Loading weights: 100%|██████████| 729/729 [00:03<00:00, 197.74it/s]


### Prompt & image helpers
Defines `resolve_image_path`, `build_prompt_and_images`, and prompt tokens. These helpers assemble the text prompt and load associated images for each example. They are used by the collator and generation utilities.

In [10]:
from PIL import Image

SYSTEM_PROMPT = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
ASSISTANT_PREFIX = "<|im_start|>assistant\n"
ASSISTANT_SUFFIX = "<|im_end|>"

def resolve_image_path(image_path):
    candidate = Path(image_path)
    if candidate.is_file():
        return candidate

    dataset_candidate = DATA_DIR / candidate
    if dataset_candidate.is_file():
        return dataset_candidate

    basename_candidate = DATA_DIR / candidate.name
    if basename_candidate.is_file():
        return basename_candidate

    for split_name in ("train", "valid", "test"):
        split_candidate = DATA_DIR / split_name / "images" / candidate.name
        if split_candidate.is_file():
            return split_candidate

    raise FileNotFoundError(f"Could not resolve image path: {image_path}")

def build_prompt_and_images(example, add_generation_prompt):
    messages = example["messages"]
    user_message = next(message for message in messages if message["role"] == "user")
    assistant_message = next((message for message in messages if message["role"] == "assistant"), None)

    image_paths = []
    text_parts = []
    for content_item in user_message["content"]:
        if content_item["type"] == "image":
            image_paths.append(resolve_image_path(content_item["image"]))
        elif content_item["type"] == "text":
            text_parts.append(content_item["text"])

    image_tokens = "".join("<|vision_start|><|image_pad|><|vision_end|>" for _ in image_paths)
    user_text = "".join(text_parts)
    prompt = SYSTEM_PROMPT + f"<|im_start|>user\n{image_tokens}{user_text}<|im_end|>\n"

    images = [Image.open(image_path).convert("RGB") for image_path in image_paths]

    if add_generation_prompt:
        return prompt + ASSISTANT_PREFIX, images

    assistant_text = "".join(
        content_item["text"]
        for content_item in (assistant_message["content"] if assistant_message is not None else [])
        if content_item.get("type") == "text"
    )
    return prompt + ASSISTANT_PREFIX + assistant_text + ASSISTANT_SUFFIX, images

### Collator (tokenization + label masking)
`collate_fn` tokenizes the full (prompt+assistant) sequence and the prompt-only sequence, then masks prompt tokens in `labels` with `-100` so the loss is computed only on the assistant-generated tokens. This cell must run before training.

In [11]:
import torch
def collate_fn(batch):
    prompt_texts = []
    full_texts = []
    images = []

    for example in batch:
        prompt_text, example_images = build_prompt_and_images(example, add_generation_prompt=True)
        full_text, _ = build_prompt_and_images(example, add_generation_prompt=False)

        prompt_texts.append(prompt_text)
        full_texts.append(full_text)
        images.append(example_images)

    tokenized_full = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True
    )

    tokenized_prompt = processor(
        text=prompt_texts,
        images=images,
        return_tensors="pt",
        padding=True
    )

    labels = tokenized_full["input_ids"].clone()

    prompt_mask_lengths = tokenized_prompt["attention_mask"].sum(dim=1)
    for i, plen in enumerate(prompt_mask_lengths.tolist()):
        if plen > 0:
            labels[i, : int(plen)] = -100

    tokenized_full["labels"] = labels

    return tokenized_full

### Training config & Trainer
This cell builds `TrainingArguments` tuned to the smoke results (safe learning rate, warmup, weight decay) and creates a `Trainer` that handles the training loop. Adjust `num_train_epochs`, `per_device_train_batch_size`, and `eval_steps` here for longer runs.

In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qwen_vl_model_full",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    dataloader_num_workers=0,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_steps=200,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    remove_unused_columns=False,
    label_names=["labels"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_grad_norm=1.0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collate_fn,
)

trainer.train()

model.save_pretrained("./qwen_vl_model_full")
processor.save_pretrained("./qwen_vl_model_full")


Step,Training Loss,Validation Loss
200,0.716490,0.021407
400,0.000102,0.023963
600,0.000046,0.024230
800,0.000036,0.024641
1000,0.000028,0.025329
1200,0.000026,0.025405
1350,0.000027,0.025431


['./qwen_vl_model_full\\processor_config.json']

### Model & processor save
After training completes, this cell saves the fine-tuned model and processor to `./qwen_vl_model_full` for later loading or inference.

In [13]:
# Manual validation pass (no Trainer callbacks)
from torch.utils.data import DataLoader
validation_loader = DataLoader(dataset["validation"], batch_size=2, shuffle=False, collate_fn=collate_fn)
model.eval()
val_losses = []
with torch.no_grad():
    for batch in validation_loader:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        out = model(**batch)
        val_losses.append(out.loss.item())

print('Validation batches:', len(val_losses))
print('Mean validation loss:', sum(val_losses)/len(val_losses) if val_losses else float('nan'))

Validation batches: 515
Mean validation loss: 0.021406816940492436


In [14]:
# Short manual sanity training: 50 steps using DataLoader + manual optimizer loop
from torch.utils.data import DataLoader
from torch.optim import AdamW

loader = DataLoader(dataset["train"], batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=0)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)

model.train()
loss_points = []
steps = 50
it = iter(loader)
for step in range(steps):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader)
        batch = next(it)

    batch = {k: v.to(model.device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    loss_points.append(loss.item())
    if (step + 1) % 10 == 0:
        print(f"step {step+1}/{steps} loss {loss.item()}")

print("Smoke steps:", len(loss_points))
print("First loss:", loss_points[0] if loss_points else None)
print("Last loss:", loss_points[-1] if loss_points else None)
print("Loss points sample:", loss_points[:10])

loss_points


step 10/50 loss 0.00045462133130058646
step 20/50 loss 0.011974489316344261
step 30/50 loss 0.0011413504835218191
step 40/50 loss 0.0018396038794890046
step 50/50 loss 0.0009073253022506833
Smoke steps: 50
First loss: 0.0011901697143912315
Last loss: 0.0009073253022506833
Loss points sample: [0.0011901697143912315, 0.0007018998730927706, 0.0006501534371636808, 0.0007892202702350914, 0.0003473600372672081, 0.0003130593686364591, 0.2294931411743164, 0.39164453744888306, 0.0002912409545388073, 0.00045462133130058646]


[0.0011901697143912315,
 0.0007018998730927706,
 0.0006501534371636808,
 0.0007892202702350914,
 0.0003473600372672081,
 0.0003130593686364591,
 0.2294931411743164,
 0.39164453744888306,
 0.0002912409545388073,
 0.00045462133130058646,
 0.00037026035715825856,
 0.0002963219885714352,
 0.00035081314854323864,
 0.000861027860082686,
 0.0003987239324487746,
 0.0008741867495700717,
 0.0004426552914083004,
 0.00042420189129188657,
 0.3878946900367737,
 0.011974489316344261,
 0.00039694327278994024,
 0.0004646516463253647,
 0.00044537242501974106,
 0.0007235673256218433,
 0.0005642292089760303,
 0.0004218537360429764,
 0.0007283837767317891,
 0.000639385893009603,
 0.0007365632918663323,
 0.0011413504835218191,
 0.001472407951951027,
 0.001238772296346724,
 0.002073263516649604,
 0.0012344485148787498,
 0.0010684531880542636,
 0.0005328116822056472,
 0.0011412013554945588,
 0.0009258489008061588,
 0.0008980749407783151,
 0.0018396038794890046,
 0.0006988033419474959,
 0.0016039141919463873,


### Manual validation pass
This final cell runs a manual no-grad validation loop over the `validation` split to compute batch-wise and mean validation loss without triggering Trainer notebook callbacks. Run this after training to measure generalization.